# Heart Disease Prediction Using Machine Learning
**Internship Project — IIDT & Blackbuck Engineers**  
**Student:** Rikka Dinesh Reddy | **Reg. No:** 21KT1A0454  
**College:** Potti Sriramulu Chalavadi Mallikarjuna Rao College of Engineering and Technology  

---

### Objective
Predict whether a person is at risk of heart disease based on various health parameters using multiple ML algorithms and compare their accuracy.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# ML Algorithms
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

print("All libraries imported successfully!")

## Step 2: Load Dataset

In [ ]:
# Load dataset
# Dataset sourced from Kaggle: Heart Disease Dataset
# https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

df = pd.read_csv('heart_disease_dataset.csv')

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

## Step 3: Dataset Overview

In [ ]:
print("Dataset Info:")
df.info()

In [ ]:
print("Statistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

In [ ]:
# Target distribution
print("Target Class Distribution:")
print(df['target'].value_counts())
print(f"\n0 = No Heart Disease, 1 = Heart Disease Present")

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
df['target'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title('Target Class Distribution')
axes[0].set_xlabel('Target (0 = No Disease, 1 = Disease)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Disease (0)', 'Disease (1)'], rotation=0)

# Pie chart
df['target'].value_counts().plot(kind='pie', ax=axes[1],
    labels=['Heart Disease', 'No Heart Disease'],
    autopct='%1.1f%%', colors=['#DD8452', '#4C72B0'])
axes[1].set_title('Target Distribution (Pie)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Age distribution by target
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='age', hue='target', bins=20, palette='Set1', kde=True)
plt.title('Age Distribution by Heart Disease Status')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend(title='Target', labels=['Disease', 'No Disease'])
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(14, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Sex vs Heart Disease
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='sex', hue='target', ax=axes[0], palette='Set2')
axes[0].set_title('Sex vs Heart Disease')
axes[0].set_xlabel('Sex (0=Female, 1=Male)')
axes[0].set_ylabel('Count')
axes[0].legend(title='Target', labels=['No Disease', 'Disease'])

sns.boxplot(data=df, x='target', y='chol', ax=axes[1], palette='Set3')
axes[1].set_title('Cholesterol Level by Heart Disease Status')
axes[1].set_xlabel('Target (0=No Disease, 1=Disease)')
axes[1].set_ylabel('Cholesterol (mg/dl)')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Age vs Max Heart Rate
plt.figure(figsize=(8, 5))
scatter = plt.scatter(df['age'], df['thalach'],
                      c=df['target'], cmap='RdYlGn', alpha=0.7, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Target (0=No Disease, 1=Disease)')
plt.xlabel('Age')
plt.ylabel('Max Heart Rate (thalach)')
plt.title('Age vs Max Heart Rate colored by Heart Disease')
plt.show()

## Step 5: Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

print("Feature Matrix Shape:", X.shape)
print("Target Vector Shape:", y.shape)
print("\nFeatures used:")
print(list(X.columns))

In [ ]:
# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size:  {X_train.shape[0]} samples")
print(f"Testing  set size:  {X_test.shape[0]} samples")

In [ ]:
# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Feature scaling applied (StandardScaler).")
print("Scaled training data — mean ≈ 0, std ≈ 1")

## Step 6: Train & Evaluate All ML Models

In [ ]:
# Define all models as described in the internship report
models = {
    'Support Vector Machine': SVC(kernel='rbf', random_state=42),
    'Naive Bayes':            GaussianNB(),
    'Decision Tree':          DecisionTreeClassifier(random_state=42),
    'Random Forest':          RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=42),
    'AdaBoost':               AdaBoostClassifier(n_estimators=100, random_state=42),
    'XGBoost':                XGBClassifier(n_estimators=100, use_label_encoder=False,
                                            eval_metric='logloss', random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred    = model.predict(X_test_scaled)
    accuracy  = accuracy_score(y_test, y_pred) * 100
    results[name] = accuracy
    print(f"{name:<30} Accuracy: {accuracy:.2f}%")

## Step 7: Accuracy Comparison (Table 1)

In [ ]:
# Build comparison DataFrame
results_df = pd.DataFrame(
    list(results.items()),
    columns=['Algorithm', 'Accuracy (%)']
).sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)

results_df.index += 1
print("\nTable 1: Accuracy Comparison of Algorithms")
print(results_df.to_string())

In [ ]:
# Bar chart — Algorithm Accuracy Comparison
colors = ['#2ecc71' if acc == max(results.values()) else '#3498db'
          for acc in results_df['Accuracy (%)']]

plt.figure(figsize=(12, 6))
bars = plt.bar(results_df['Algorithm'], results_df['Accuracy (%)'], color=colors, edgecolor='black')

for bar, val in zip(bars, results_df['Accuracy (%)']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.2f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.title('Algorithm Accuracy Comparison — Heart Disease Prediction', fontsize=14)
plt.xlabel('Algorithm')
plt.ylabel('Accuracy (%)')
plt.ylim(60, 105)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print("Green bar = Best performing model")

## Step 8: Best Model — Random Forest (Detailed Analysis)

In [ ]:
# Retrain best model
best_model = RandomForestClassifier(n_estimators=100, random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred_best = best_model.predict(X_test_scaled)

print("Best Model: Random Forest")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_best)*100:.2f}%")
print("\nTable 2: Classification Report")
print(classification_report(y_test, y_pred_best, target_names=['No Disease (0)', 'Disease (1)']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['No Disease (0)', 'Disease (1)'])
disp.plot(cmap='Blues', colorbar=False)
plt.title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feature_importances = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feature_importances.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Feature Importances — Random Forest')
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print("\nTop 5 Important Features:")
print(feature_importances.head())

## Step 9: Cross-Validation (5-Fold)

In [ ]:
from sklearn.model_selection import StratifiedKFold

rf_cv = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rf_cv, X, y, cv=StratifiedKFold(n_splits=5), scoring='accuracy')

print("5-Fold Cross-Validation — Random Forest")
print(f"Scores per fold : {[f'{s*100:.2f}%' for s in cv_scores]}")
print(f"Mean Accuracy   : {cv_scores.mean()*100:.2f}%")
print(f"Std Deviation   : {cv_scores.std()*100:.2f}%")

## Step 10: Predict on New Patient Data

In [ ]:
# Input a new patient's data to predict heart disease risk
# Format: [age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal]

new_patient = np.array([[63, 1, 3, 145, 233, 1, 0, 150, 0, 2.3, 0, 0, 1]])

new_patient_scaled = scaler.transform(new_patient)
prediction = best_model.predict(new_patient_scaled)
probability = best_model.predict_proba(new_patient_scaled)

print("Patient Data Input:")
print(pd.DataFrame(new_patient, columns=X.columns).to_string(index=False))

print(f"\nPrediction : {'Heart Disease PRESENT ⚠️' if prediction[0] == 1 else 'No Heart Disease ✅'}")
print(f"Probability: No Disease = {probability[0][0]*100:.1f}% | Disease = {probability[0][1]*100:.1f}%")

## Conclusion

This project successfully demonstrated the use of seven machine learning algorithms to predict heart disease:

| Algorithm | Accuracy |
|---|---|
| **Random Forest** | **~98.54%** |
| **Decision Tree** | **~98.54%** |
| **XGBoost** | **~98.54%** |
| SVM | ~88.78% |
| AdaBoost | ~81.46% |
| Naive Bayes | ~80.00% |
| Logistic Regression | ~79.51% |

**Random Forest** was selected as the best model due to its highest accuracy and robustness, confirmed by 5-fold cross-validation. The model can assist in early identification of individuals at risk of heart disease, enabling timely medical intervention.

---
*Internship Project — IIDT & Blackbuck Engineers, Tirupati, Andhra Pradesh (2024)*